# **Benchmark Model**

## Libraries

In [ ]:
# Libraries
import sys
import pandas as pd
from tensorflow import keras
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.image import load_img, img_to_array, smart_resize, ImageDataGenerator

from tensorflow.keras.applications.efficientnet import EfficientNetB0, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau


sys.path.append('../scripts')  # Add root folder of project

from display_utils import show_image
from constants import METADATA_PATH, IMAGE_DIR, IMAGE_SIZE_STANDARD
from preprocess_utils import resize_image

In [ ]:
# Setting options
pd.set_option('display.max_rows', None)

## Data Loading

In [ ]:
# Load the metadata
df = pd.read_csv(f'../{METADATA_PATH}')

# Load the IsAnimal data
is_animal = pd.read_csv(f'../{IMAGE_DIR}/answers.txt', delimiter=',', header=None, names=['image_id', 'is_animal'])

# Join the two dataframes
data = pd.merge(
    df
    ,is_animal
    ,left_on='rare_species_id'
    ,right_on='image_id'
    ,how='inner'
)

In [ ]:
# Creating a sample
train_df, sample_df = train_test_split(
    data,
    test_size=0.3,
    stratify=df['family'],
    random_state=20
)

## Data Preprocessing

In [ ]:
# Performing the splits
train_df, test_df = train_test_split(sample_df, test_size=0.2, stratify=sample_df['family'], random_state=20)  # Create test set
train_df, val_df = train_test_split(train_df, test_size=0.15, stratify=train_df['family'], random_state=20)  # Create validation set

In [ ]:
# Encoding the target
enc = LabelEncoder()

train_df['family'] = enc.fit_transform(train_df['family'])
val_df['family'] = enc.transform(val_df['family'])
test_df['family'] = enc.transform(test_df['family'])

In [ ]:
# Image resizing target dimensions
IMG_SIZE = 224
BATCH_SIZE = 8

In [ ]:
# Defining functions to resize the images
def smart_resize_img(file_path, target_size=(IMG_SIZE, IMG_SIZE)):
    img = load_img(f'../{IMAGE_DIR}/{file_path}')  # Load image
    img_array = img_to_array(img)  # Convert to array
    resized_img = smart_resize(img_array, target_size)  # Resize image
    resized_img /= 255.0
    
    return resized_img

# Custom generator that uses smart_resize
def smart_resize_generator(dataframe, target, batch_size=BATCH_SIZE, target_size=(IMG_SIZE, IMG_SIZE)):
    while True:
        # Iterate through the dataframe in batches
        for i in range(0, len(dataframe), batch_size):
            batch_df = dataframe.iloc[i:i+batch_size]
            
            # Prepare the batch of images and labels
            batch_images = np.array([smart_resize_img(file_path, target_size) for file_path in batch_df['file_path']])
            batch_labels = np.array(batch_df[target])
            
            yield batch_images, batch_labels

In [ ]:
# Resizing the images
is_animal_train_generator = smart_resize_generator(train_df, 'is_animal', batch_size=BATCH_SIZE, target_size=(IMG_SIZE, IMG_SIZE))
print('train resizing completed')
is_animal_val_generator = smart_resize_generator(val_df, 'is_animal', batch_size=BATCH_SIZE, target_size=(IMG_SIZE, IMG_SIZE))
print('validation resizing completed')
is_animal_test_generator = smart_resize_generator(test_df, 'is_animal', batch_size=BATCH_SIZE, target_size=(IMG_SIZE, IMG_SIZE))
print('test resizing completed')

In [ ]:
# Build the model

# Set the input
input_tensor = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

# Load the pre-trained model
base_model = EfficientNetB0(include_top=False, weights='imagenet', input_tensor=input_tensor)

# Add necessary layers
x = GlobalAveragePooling2D()(base_model.output)
output = Dense(1, activation='sigmoid')(x)  # Binary classification

model = Model(inputs=input_tensor, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy', 'precision', 'recall']
)

In [ ]:
# Fit the model
model.fit(
    is_animal_train_generator,
    validation_data=is_animal_val_generator,
    epochs=1,
    steps_per_epoch=int(np.ceil(len(train_df) / BATCH_SIZE)),
    callbacks=[
        EarlyStopping(patience=3, restore_best_weights=True),
        # ModelCheckpoint('best_model.h5', save_best_only=True),
        ReduceLROnPlateau(patience=2, factor=0.5, verbose=1)
    ],
    verbose=1
)

In [ ]:
# Evaluate on test set
model.evaluate(is_animal_test_generator)